# 02 — Train a DQN Model Offline

Load a stored transition dataset and train a MOUSE model entirely from replayed data, with no live environment in the training loop.

A MOUSE model is assembled from three independent pieces:

- `StepEmbedder` converts each `(action, observation, reward, done)` step into a token sequence the backbone can process.
- `Qwen3Backbone` reads those tokens with a transformer and builds up context over the sequence.
- `DiscreteActionValueHead` maps the backbone's output to one Q-value per action.

`Qwen3Backbone` downloads `Qwen/Qwen3-0.6B` from the Hub on first run and keeps only the first `num_layers` transformer layers. Its `hidden_dim` (1024 for this checkpoint) is the single number that connects all three pieces — no manual dimension matching required.

In [13]:
import torch

from mouse_core.data import (
    DataLoader,
    SequenceAugmenter,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import StepEmbedder
from mouse_core.models.heads import DiscreteActionValueHead

# Hub dataset repo containing collected transition stores.
DATASET_ID = "mouse-example-dataset"

# Shared Hub model repo used by the inference notebook.
# Offline and online training both push here; inference loads whichever checkpoint was pushed last.
MODEL_ID = "mouse-example-model"

# FrozenLake has four discrete actions, matching the DQN head width.
MAX_ACTIONS = 4

# All env instances use 4x4 maps (16 discrete states, 0-15).
MAX_OBS_DISCRETE = 16

TRAIN_STEPS = 4000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load data

`load_stores_from_hub` downloads the named dataset repo and reconstructs the four `Datastore` objects — one per stored maze stream.

In [14]:
stores = load_stores_from_hub(DATASET_ID, split="train")
for s in stores:
    print(s)

README.md:   0%|          | 0.00/2.60k [00:00<?, ?B/s]

data/proc_frozenlake_0/train-00000-of-00(…):   0%|          | 0.00/107k [00:00<?, ?B/s]

data/proc_frozenlake_1/train-00000-of-00(…):   0%|          | 0.00/107k [00:00<?, ?B/s]

data/proc_frozenlake_2/train-00000-of-00(…):   0%|          | 0.00/107k [00:00<?, ?B/s]

data/proc_frozenlake_3/train-00000-of-00(…):   0%|          | 0.00/107k [00:00<?, ?B/s]

Datastore(name='proc_frozenlake_0', steps=1500)
Datastore(name='proc_frozenlake_1', steps=1500)
Datastore(name='proc_frozenlake_2', steps=1500)
Datastore(name='proc_frozenlake_3', steps=1500)


## DataLoader and augmentation

`DataLoader` slices one or more `Datastore`s into fixed-length sequence batches and mixes them automatically — no manual merging required.

- **`sequence_length=16`** — each item in a batch is 16 consecutive steps from a single env instance.
- **`batch_size=32`** — 32 sequences per call to `next_batch()`.
- **`prefetch=4`** — pre-loads 4 batches in the background to keep the GPU fed.

`next_batch()` returns a `list[list[dict]]` of shape `[batch][sequence]` — one inner list per sequence, one dict per step. Passing `augmenter=augment` makes `DataLoader` run `SequenceAugmenter` inside the sampling path. Augmentation modality specs use `field` plus an augmentation `type` such as `discrete`, `linear`, or `image`.

In [15]:
augmenter = SequenceAugmenter(
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
    seed=0,
)

loader = DataLoader(
    stores,
    sequence_length=128,
    batch_size=4,
    prefetch=4,
    num_workers=0,
    augmenter=augmenter,
)

## Build the model

All three components are sized by `backbone.hidden_dim`, so they connect without any manual dimension matching.

**`StepEmbedder`** — each entry in `modalities` describes one field from the step dict and which `type` to use:
- `"discrete"` — looks up a learned vector from a small vocabulary. Use for integer-valued fields like actions, observations, and done codes.
- `"rff"` — embeds a continuous scalar using random Fourier features. Use for rewards or any float value.
- `"learnable"` — a free learnable token with no input, giving the model scratch space it can write to without consuming an observation slot.

**`DiscreteActionValueHead`** — a small MLP that predicts one Q-value per action. `scale=0.01` initialises output weights near zero so Q-values start small and stable.

**`Model`** — wraps encoder, backbone, and head(s) into a single object with a unified `forward` call.

In [16]:
backbone = Qwen3Backbone(pretrained="Qwen/Qwen3-0.6B", num_layers=2)

encoder = StepEmbedder(
    hidden_dim=backbone.hidden_dim,
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "std": 0.02,
            "tokens": 1
        },
        {
            "field": "reward",
            "type": "rff",
            "std": 0.02,
            "in_min": 0.001,
            "in_max": 1000.0,
            "tokens": 1
        },
        {
            "field": "done",
            "type": "discrete",
            "vocab_size": 5,
            "std": 0.02,
            "tokens": 1
        },
        {
            "type": "learnable",
            "tokens": 1
        },
    ],
    concat_modalities=False,
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.01,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
print(model)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Model(
  (encoder): StepEmbedder(
    (type_embedder): TypeEmbedder(
      (embed): ScaledEmbedding(64, 1024)
    )
    (modality_embedders): ModuleDict(
      (action): DiscreteEmbedder(
        (embed): ScaledEmbedding(4, 1024)
      )
      (observation): DiscreteEmbedder(
        (embed): ScaledEmbedding(16, 1024)
      )
      (reward): ScalarRFFEmbedder(
        (rff): RandomFourierFeatures()
      )
      (done): DiscreteEmbedder(
        (embed): ScaledEmbedding(5, 1024)
      )
      (__learnable_4): LearnableEmbedder()
    )
  )
  (backbone): Qwen3Backbone(
    (model): Qwen3Model(
      (embed_tokens): Embedding(1, 1024)
      (layers): ModuleList(
        (0-1): 2 x Qwen3DecoderLayer(
          (self_attn): Qwen3Attention(
            (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
            (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
            (o_proj):

## Train

Each iteration:
1. Sample an already-augmented batch of transition sequences from the data loader.
2. Run both the online network and a frozen target network to get Q-value predictions.
3. Compute the Bellman loss — `objective(objective_data, predictions)` returns `(loss, metrics)`.
4. Back-propagate, clip gradients, and step the optimizer.
5. Slowly copy online weights → target network (Polyak/EMA update) for stable TD targets.

### Discount factors and done codes

MOUSE uses five `done` codes. `DqnObjective` maps each to its own bootstrap discount:

- `0`: running, using `gamma_step`.
- `1`: episode done within a task, using `gamma_episode_terminal`.
- `2`: episode truncated within a task, using `gamma_episode_truncated`.
- `3`: task done, using `gamma_task_terminal`.
- `4`: task truncated, using `gamma_task_truncated`.

The example dataset is collected with `episodes_per_task=0`, so all episodes belong to one ongoing task. We bootstrap through every done code with `0.99`, which keeps the target connected across episode and task markers instead of cutting it off.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
objective = DqnObjective(
    gamma_step=1.0,
    gamma_episode_terminal=0.99,   # bootstrap through episode boundaries
    gamma_episode_truncated=0.99,
    gamma_task_terminal=0.99,      # bootstrap through task boundaries too
    gamma_task_truncated=0.99,
    tau=0.005,
)

for step in range(TRAIN_STEPS):

    batch = loader.next_batch()

    predictions, objective_data, _ = model(batch)
    loss, _ = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)

    if step % 100 == 0:
        print(f"step={step}  loss={loss}")

loader.close()
print("Training finished.")

step=0  loss=0.106290303170681
step=100  loss=0.07498825341463089
step=200  loss=0.05949680134654045
step=300  loss=0.05748466029763222
step=400  loss=0.045591067522764206
step=500  loss=0.034426022320985794
step=600  loss=0.026271525770425797
step=700  loss=0.03994853049516678
step=800  loss=0.04366312921047211
step=900  loss=0.03761032968759537
step=1000  loss=0.04746512696146965
step=1100  loss=0.0347779206931591
step=1200  loss=0.037999674677848816
step=1300  loss=0.045071713626384735
step=1400  loss=0.04517083615064621
step=1500  loss=0.04460875317454338
step=1600  loss=0.039869096130132675
step=1700  loss=0.04416471719741821
step=1800  loss=0.052262164652347565
step=1900  loss=0.04279755428433418
step=2000  loss=0.056666016578674316
step=2100  loss=0.044859372079372406
step=2200  loss=0.05108048394322395
step=2300  loss=0.05058874562382698
step=2400  loss=0.048468347638845444
step=2500  loss=0.048667751252651215
step=2600  loss=0.03559813275933266
step=2700  loss=0.04528937861323

## Push to the Hub

Save the offline-trained model to the shared Hub repo under `MODEL_ID`. The inference notebook loads the current checkpoint from that repo, so whichever training notebook you push last is the model it evaluates.

In [18]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")

Pushed to https://huggingface.co/micahr234/mouse-example-model
